# MACD basic trading strategy
Based on the following video\
【A股：这才是MACD的极致用法，我整整读了10遍，太精辟透彻了！-哔哩哔哩】 https://b23.tv/1hVHecU \
Upper bound is defined as the max of the day where 

In [33]:
# Cell 1: Import modules and load/process data
import pandas as pd
import sys
import os
import plotly.graph_objects as go
import pandas_ta as ta

class colors:
       RED = '\033[91m'
       GREEN = '\033[92m'
       YELLOW = '\033[93m'
       BLUE = '\033[94m'
       PINK = '\033[95m'
       CYAN = '\033[96m'
       ENDC = '\033[0m' # Reset to default

# Get the absolute path to the directory containing the current notebook
# and then move up one level to the project root
module_path = os.path.abspath(os.path.join('..'))

if module_path not in sys.path:
       sys.path.append(module_path)

# Now you can import from the modules folder
from modules.data_processor import FeatureFactory
from modules.environment import StockTradingEnv
from modules.model_agent import TradingAgent
from modules.rewards import RewardFunction
# Assuming evaluator.py exists for Module E
from modules.evaluator import Evaluator  # If not present, skip evaluation

# Load raw data
raw_df = pd.read_csv('../data/2454_2015-2025.csv')

# Process data with Module A
factory = FeatureFactory(normalize = False)
df = factory.process(raw_df)
print("Data processed with FeatureFactory.")
print(df.head(5))
df.to_csv('../data/2454_2015-2025_processed.csv', index=False)
print(df.columns)
initial_cash = 10000000
holdings = 0
upper_bound = None
lower_bound = None
PPO_THRESHOLD = 0.5
STOP_LOSS_THRESHOLD = 0.1
asset_snapshot = initial_cash

state = -1 # -1 for no position, 0 for holding cash, 1 for holding stock
for index in range(len(df.index)):
       if (index < 4):
              continue
       center_date = index-2
       three_candles_pos = df.iloc[center_date]['MACDh_12_26_9'] > 0 and df.iloc[center_date -1]['MACDh_12_26_9'] > 0 and df.iloc[center_date +1]['MACDh_12_26_9'] > 0
       three_candles_neg = df.iloc[center_date]['MACDh_12_26_9'] < 0 and df.iloc[center_date -1]['MACDh_12_26_9'] < 0 and df.iloc[center_date +1]['MACDh_12_26_9'] < 0
       if three_candles_pos:
              if ( max(df.iloc[center_date -1 : center_date+2]['MACDh_12_26_9']) == df.iloc[center_date]['MACDh_12_26_9'] and df.iloc[center_date]['PPOh_12_26_9'] > PPO_THRESHOLD ):
                     lower_bound = df.iloc[center_date]['min']
       elif three_candles_neg:
              if ( min(df.iloc[center_date -1 : center_date+2]['MACDh_12_26_9']) == df.iloc[center_date]['MACDh_12_26_9'] and df.iloc[center_date]['PPOh_12_26_9'] < -PPO_THRESHOLD ):
                     upper_bound = df.iloc[center_date]['max']
       asset = initial_cash + holdings * df.iloc[index]['close']
       
       if lower_bound is not None and df.iloc[index]['close'] < lower_bound and state == 1:
              initial_cash += holdings * df.iloc[index]['close']
              holdings -= holdings
              state = 0
              print(f"{colors.RED}Sell signal at {df.index[index]}: price {df.iloc[index]['close']} crossed below lower bound {lower_bound}{colors.ENDC}")
              print(f"Sell price: {df.iloc[index]['close']}")
              print(f"Holdings after sell: {holdings}, Cash: {initial_cash}")
              print(f"total asset: {initial_cash + holdings * df.iloc[index]['close']}") 
              upper_bound = None
       if upper_bound is not None and df.iloc[index]['close'] > upper_bound and state !=1:
              holdings += initial_cash // df.iloc[index]['close']
              initial_cash -= (initial_cash // df.iloc[index]['close']) * df.iloc[index]['close']
              state = 1
              print(f"{colors.GREEN}Buy signal at {df.index[index]}: price {df.iloc[index]['close']} crossed above upper bound {upper_bound}{colors.ENDC}")
              print(f"Buy price: {df.iloc[index]['close']}")
              print(f"Holdings after buy: {holdings}, Cash: {initial_cash}")
              print(f"total asset: {initial_cash + holdings * df.iloc[index]['close']}")
              lower_bound = None

Data processed with FeatureFactory.
            stock_id  Trading_Volume  Trading_money   open    max    min  \
date                                                                       
2015-05-22      2454         8993585     3679381954  405.0  412.0  403.0   
2015-05-25      2454         6774725     2718607575  405.5  409.0  396.5   
2015-05-26      2454         3161817     1265169800  401.0  403.0  398.0   
2015-05-27      2454         4852855     1942833000  399.5  403.0  399.5   
2015-05-28      2454         9060437     3708076429  410.0  412.0  406.0   

            close  spread  Trading_turnover     RSI_14  MACD_12_26_9  \
date                                                                   
2015-05-22  410.0    10.0              5823  64.767440     -3.729389   
2015-05-25  397.5   -12.5              4794  56.452894     -3.198459   
2015-05-26  399.5     2.0              2554  57.395313     -2.586495   
2015-05-27  400.0     0.5              2926  57.642112     -2.037674   